# Strategy: Patron Comportamental

## Situacion planteada
Una entidad financiera ofrece creditos asociados a diferentes tarjetas. Cada tarjeta aplica una regla financiera distinta para calcular los intereses:

- La tarjeta Clasica usa interes simple mensual del 2%.
- La tarjeta Platinum usa interes compuesto mensual del 1.2%.

El credito debe calcular la cuota sin conocer los detalles de cada formula. El problema no consiste solamente en cambiar una tasa: tambien cambia el algoritmo financiero que produce el total de intereses.

El patron **Strategy** permite encapsular cada formula como una estrategia intercambiable. El tipo de tarjeta puede cambiar en tiempo de ejecucion sin modificar la clase `Credito`.

In [16]:
class CreditoSinStrategy:
    def __init__(self, capital, plazo_meses, tipo_tarjeta):
        self.capital = capital
        self.plazo_meses = plazo_meses
        self.tipo_tarjeta = tipo_tarjeta

    def calcular_intereses(self):
        if self.tipo_tarjeta == "clasica":
            tasa_mensual = 0.02
            return self.capital * tasa_mensual * self.plazo_meses
        elif self.tipo_tarjeta == "platinum":
            tasa_mensual = 0.012
            total_con_intereses = self.capital * (1 + tasa_mensual) ** self.plazo_meses
            return total_con_intereses - self.capital
        else:
            raise "Tipo de tarjeta no soportado"

    def calcular_cuota_mensual(self):
        total = self.capital + self.calcular_intereses()
        return round(total / self.plazo_meses, 2)

credito_clasica_sin = CreditoSinStrategy(2000000, 12, "clasica")
credito_platinum_sin = CreditoSinStrategy(2000000, 12, "platinum")
print("Cuota Clasica sin Strategy:", credito_clasica_sin.calcular_cuota_mensual())
print("Cuota Platinum sin Strategy:", credito_platinum_sin.calcular_cuota_mensual())

Cuota Clasica sin Strategy: 206666.67
Cuota Platinum sin Strategy: 192315.77


### Problema observado

- `CreditoSinStrategy` conoce las reglas financieras de todas las tarjetas.
- La clase mezcla dos algoritmos distintos: interes simple e interes compuesto.
- Agregar una tarjeta con una nueva formula exige editar `calcular_intereses`.
- La seleccion por texto y los condicionales hacen crecer la clase cada vez que aparece un nuevo producto.

La clase necesita variar el algoritmo, pero su responsabilidad principal deberia ser administrar el credito y calcular la cuota.

## Con patron Strategy

La interfaz `EstrategiaInteres` define el contrato comun. `InteresTarjetaClasica` encapsula el interes simple y `InteresTarjetaPlatinum` encapsula el interes compuesto. `Credito` recibe una estrategia por inyeccion y no conoce la formula interna.

In [14]:
from abc import ABC, abstractmethod

class EstrategiaInteres(ABC):
    @abstractmethod
    def calcular(self, capital, plazo_meses):
        raise NotImplementedError

class InteresTarjetaClasica(EstrategiaInteres):
    def calcular(self, capital, plazo_meses):
        tasa_mensual = 0.02
        return capital * tasa_mensual * plazo_meses

class InteresTarjetaPlatinum(EstrategiaInteres):
    def calcular(self, capital, plazo_meses):
        tasa_mensual = 0.012
        total_con_intereses = capital * (1 + tasa_mensual) ** plazo_meses
        return total_con_intereses - capital

class Credito:
    def __init__(self, capital, plazo_meses, estrategia):
        self.capital = capital
        self.plazo_meses = plazo_meses
        self.estrategia = estrategia

    def cambiar_estrategia(self, estrategia):
        self.estrategia = estrategia

    def calcular_intereses(self):
        return self.estrategia.calcular(self.capital, self.plazo_meses)

    def calcular_cuota_mensual(self):
        total = self.capital + self.calcular_intereses()
        return round(total / self.plazo_meses, 2)

In [15]:
credito = Credito(2000000, 12, InteresTarjetaClasica())
print("Cuota con tarjeta Clasica:", credito.calcular_cuota_mensual())

credito.cambiar_estrategia(InteresTarjetaPlatinum())
print("Cuota con tarjeta Platinum:", credito.calcular_cuota_mensual())

Cuota con tarjeta Clasica: 206666.67
Cuota con tarjeta Platinum: 192315.77


## UML del ejemplo

```plantuml
@startuml
interface EstrategiaInteres {
    + calcular(capital, plazo_meses)
}
class InteresTarjetaClasica {
    + calcular(capital, plazo_meses)
}
class InteresTarjetaPlatinum {
    + calcular(capital, plazo_meses)
}
class Credito {
    - capital
    - plazo_meses
    - estrategia
    + cambiar_estrategia(estrategia)
    + calcular_intereses()
    + calcular_cuota_mensual()
}
EstrategiaInteres <|.. InteresTarjetaClasica
EstrategiaInteres <|.. InteresTarjetaPlatinum
Credito --> EstrategiaInteres
@enduml
```

## Por que Strategy y no otro patron?

Se eligio **Strategy** porque el problema contiene una familia de algoritmos de calculo que debe poder cambiarse sin modificar el contexto:

- `InteresTarjetaClasica` calcula interes simple.
- `InteresTarjetaPlatinum` calcula interes compuesto.

No se trata solamente de guardar una tasa distinta. Las dos estrategias aplican formulas diferentes y producen comportamientos distintos para el mismo capital y plazo. Ademas, `Credito` puede cambiar de estrategia en tiempo de ejecucion mediante `cambiar_estrategia`.

Si todas las tarjetas utilizaran la misma formula y solo cambiara la tasa, bastaria con configurar un atributo. Strategy se justifica aqui porque las reglas financieras pueden crecer y variar de manera independiente.

**Factory Method** podria ayudar a crear objetos de tarjeta, pero no es el problema principal: aqui necesitamos variar un comportamiento de calculo. **Bridge** tampoco es la mejor opcion porque no se estan separando dos jerarquias independientes; se esta encapsulando una familia de algoritmos intercambiables.

La solucion permite agregar `InteresTarjetaOro` con una nueva formula sin modificar `Credito` ni las estrategias existentes.

## Conclusion

La version sin patron concentra dos algoritmos financieros y sus decisiones en una sola clase con condicionales. Con Strategy, `Credito` depende de la abstraccion `EstrategiaInteres`, mientras cada tarjeta encapsula su propia formula. Esto mejora la extensibilidad y permite cambiar el comportamiento en tiempo de ejecucion.